# TechArena fixed five-seed shared-item GPU experiment
Exploratory reused cohorts. No CSV export or official-score claim. Enable T4 GPU and use the existing neural-history dataset.
Use this standalone notebook for Save & Run All. Preserve audit and model ZIPs from the Output panel before ending an interactive session. These ZIPs are experiment evidence, not competition submissions.


In [ ]:
from pathlib import Path
import hashlib, importlib.util
module_path=Path("/kaggle/working/tied_history_gpu_bag.py")
module_path.write_text("\"\"\"Five predeclared tied-history GPU seeds using the existing neural bundle.\n\nIndependent full-prefix reconstruction, matched seed7201 control, fixed 5% rank\nblend. No validation early stopping, seed selection, or competition CSV export.\nThe cohorts were already reused and cannot support promotion by themselves.\n\"\"\"\nimport gc\nimport hashlib\nimport json\nimport subprocess\nimport time\nfrom bisect import bisect_left\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom scipy.stats import rankdata\nfrom sklearn.metrics import roc_auc_score\n\nRECIPE = dict(seeds=[7201, 7202, 7203, 7204, 7205], steps=2000,\n              batch_size=1024, auxiliary_batch_size=512, auxiliary_weight=0.1,\n              dimensions=32, learning_rate=0.01, sampled_vector_regularization=0.002,\n              negatives=9, context_limit=100, blend_weight=0.05,\n              minimum_primary_adjusted_gain=0.0005, cpu_threads=2)\nEXPECTED = {\n    'events.npz': '3e2d628a35a9e69c6c75b0a98da28ee9dab96857209b2d16def26ce7d5c8249e',\n    'val5.npz': '6f30b7c3b825a3481006b83f02b81d617b113fb40d4898f34f504b6b9473fd3d',\n    'val6.npz': '963b8f1c8ff1edbeb88955cc087ddeccf8a4784472d488b1270b2eb6deff5d8d',\n}\nPREPARED = {\n    6: {\n        'fit_h': '91735dca1e65c475254bb60ed09cd8802f45bb9ca152104c6a277cd9e0d9ed59',\n        'fit_i': 'a4fcee5ec2b954efb27dc3a6581f85fb0994d3b4494891d1859b6c65a5a4e64d',\n        'fit_y': 'c3526f5dc44e90e58c25d18c16415a30ef1b144bfb21c8fd0d388535c993cacc',\n        'aux_h': '30ff4574d87fa0a88572401b851b68b84f9b4266cfbd017173542174c2dfa868',\n        'aux_target': 'ca8539e97c620ebcadcfb609876788f8615d19fd813ef604266872e0bea6844e',\n        'val_h': '652d5c750a633e140f14d20a07e004c433ba145a29a3265f3d75a8bbcd4e25f2',\n        'val_i': 'a6fe85b78545ee78477ae90dc02562761a34b0c5100f15ccef2ddf8c98c13019',\n    },\n    5: {\n        'fit_h': 'f1c6da162f62802a6f7ab09358c096e6b54bba2d96e04b04923a0a9841251b39',\n        'fit_i': 'c97faba33966617ada0873354c13208bb8e256c78e9933f6e6a77ab59a56a419',\n        'fit_y': '4b0f10af758de988b44326a64e6e5bb4672f4ae0581c0a2ecdc97242c8ee0b33',\n        'aux_h': '724f9d7a0c5b0a6407249ca53b5613d749149eed011a531161f9f1a622981a36',\n        'aux_target': '456a892b8b736f3e4ee83b76d872656c964f90491ea85b1b99285eb3bf94c994',\n        'val_h': '3819af854a71b96674f4c936d331864fbe81eda65d4b04fc861643d8444e0d05',\n        'val_i': '6915c0db249ea82079daf33e1fac4db1d677ede77156de01ec24fab0e0daaeb4',\n    },\n}\n\n\ndef verify_prepared(prepared, day):\n    checked = {}\n    for name, expected in PREPARED[day].items():\n        view = memoryview(np.ascontiguousarray(prepared[name])).cast('B')\n        value = hashlib.sha256()\n        for offset in range(0, len(view), 4*1024*1024):\n            value.update(view[offset:offset+4*1024*1024])\n        checked[name] = value.hexdigest()\n        assert checked[name] == expected, f'CPU context mismatch: day{day} {name}'\n    return checked\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open('rb') as stream:\n        for block in iter(lambda: stream.read(4*1024*1024), b''):\n            h.update(block)\n    return h.hexdigest()\n\n\nclass BagFactor(nn.Module):\n    def __init__(self, n_items, dimensions=32):\n        super().__init__()\n        self.items = nn.Embedding(n_items, dimensions)\n        self.bias = nn.Parameter(torch.zeros(n_items))\n        self.global_bias = nn.Parameter(torch.tensor(-2.2))\n        nn.init.normal_(self.items.weight, std=0.005)\n\n    def forward_and_penalty(self, history, candidates):\n        valid = history >= 0\n        vectors = self.items(history.clamp_min(0))*valid.unsqueeze(-1)\n        count = valid.sum(1).clamp_min(1).unsqueeze(-1)\n        profile = vectors.sum(1)/count\n        candidate_vectors = self.items(candidates)\n        bias = self.bias[candidates]\n        logits = ((profile.unsqueeze(1)*candidate_vectors).sum(-1)+bias+self.global_bias)\n        penalty = (vectors.square().sum()/valid.sum().clamp_min(1)\n                   + candidate_vectors.square().sum(-1).mean()+bias.square().mean())\n        return logits, penalty\n\n    def forward(self, history, candidates):\n        return self.forward_and_penalty(history, candidates)[0]\n\n\ndef restore_timestamps(data):\n    valid = data['hist'] >= 0\n    gaps = data['gaps'].astype('float64')\n    if not np.isfinite(gaps).all() or (gaps < 0).any():\n        raise ValueError('Invalid history ages')\n    stamps = np.rint(data['timestamp'].astype('int64')[:, None]-gaps*86400).astype('int64')\n    if not np.all((stamps < data['timestamp'][:, None]) | ~valid):\n        raise ValueError('Noncausal reconstructed history')\n    return stamps\n\n\ndef pad(clicks):\n    result = np.full(RECIPE['context_limit'], -1, np.int32)\n    chosen = sorted(clicks, reverse=True)[:len(result)]\n    result[:len(chosen)] = [item for _, item in chosen]\n    return result\n\n\ndef prepare(data, stamps, validation, cutoff, progress=True):\n    \"\"\"Reconstruct identical CPU fitting/auxiliary contexts from raw event IDs.\"\"\"\n    fit_ids = np.flatnonzero(data['timestamp'] < cutoff)\n    ledger = defaultdict(set)\n    for n, q in enumerate(fit_ids):\n        user = int(data['users'][q])\n        for item, timestamp in zip(data['hist'][q], stamps[q]):\n            if item >= 0:\n                assert timestamp < data['timestamp'][q] < cutoff\n                ledger[user].add((int(timestamp), int(item)))\n        target = int(data['target'][q])\n        if target < 10:\n            ledger[user].add((int(data['timestamp'][q]), int(data['items'][q, target])))\n        if progress and n and n % 100000 == 0:\n            print(json.dumps(dict(stage='ledger', fitting_events=n)), flush=True)\n    ledger = {user: tuple(sorted(clicks)) for user, clicks in ledger.items()}\n    def prior(user, timestamp):\n        clicks = ledger.get(user, ())\n        stop = bisect_left([t for t, _ in clicks], timestamp)\n        return clicks[max(0, stop-RECIPE['context_limit']):stop]\n    fit_h = np.empty((len(fit_ids), RECIPE['context_limit']), np.int32)\n    for n, q in enumerate(fit_ids):\n        fit_h[n] = pad(prior(int(data['users'][q]), int(data['timestamp'][q])))\n    targets, auxiliary = [], []\n    for clicks in ledger.values():\n        for target in clicks:\n            others = [value for value in clicks if value != target and value[0] < cutoff]\n            if others:\n                targets.append(target[1])\n                auxiliary.append(pad(others))\n    auxiliary = np.asarray(auxiliary, dtype=np.int32)\n    targets = np.asarray(targets, dtype=np.int32)\n    ids = validation['indices'].astype('int64')\n    future = np.flatnonzero((data['timestamp'] >= cutoff) & (data['timestamp'] < cutoff+86400))\n    _, first = np.unique(data['users'][future], return_index=True)\n    assert np.array_equal(np.sort(future[first]), np.sort(ids))\n    y = (data['target'][ids, None] == np.arange(10)).astype('float32')\n    assert np.array_equal(y.ravel(), validation['y'])\n    val_h = np.empty((len(ids), RECIPE['context_limit']), np.int32)\n    for n, q in enumerate(ids):\n        clicks = set(prior(int(data['users'][q]), int(data['timestamp'][q])))\n        clicks.update((int(t), int(i)) for i, t in zip(data['hist'][q], stamps[q]) if i >= 0)\n        val_h[n] = pad(clicks)\n    n_items = max(int(data['items'].max()), int(data['hist'].max()))+1\n    counts = np.bincount(data['items'][fit_ids].ravel(), minlength=n_items).astype('float64')\n    probability = counts/counts.sum()\n    fit_y = (data['target'][fit_ids, None] == np.arange(10)).astype('float32')\n    return dict(fit_h=fit_h, fit_i=data['items'][fit_ids], fit_y=fit_y,\n                aux_h=auxiliary, aux_target=targets, val_h=val_h,\n                val_i=data['items'][ids], exposure_probability=probability, n_items=n_items)\n\n\ndef load(root):\n    for name, expected in EXPECTED.items():\n        assert digest(root/name) == expected, f'Unexpected bundle version: {name}'\n    with np.load(root/'events.npz') as z:\n        data = {key: z[key].copy() for key in z.files}\n    assert np.isin(data['target'], np.arange(11)).all()\n    assert data['items'].shape == (len(data['timestamp']), 10)\n    assert np.all(np.diff(data['timestamp'].astype('int64')) >= 0)\n    stamps = restore_timestamps(data)\n    return data, stamps\n\n\ndef one_seed(prepared, seed, output, device):\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    rng = np.random.default_rng(seed)\n    model = BagFactor(prepared['n_items'], RECIPE['dimensions']).to(device)\n    with torch.no_grad():\n        rate = float(prepared['fit_y'].mean().item())\n        model.global_bias.fill_(np.log(rate/(1-rate)))\n    optimizer = torch.optim.Adam(model.parameters(), lr=RECIPE['learning_rate'])\n    started = time.monotonic()\n    for step in range(RECIPE['steps']):\n        ii = rng.integers(len(prepared['fit_h']), size=RECIPE['batch_size'])\n        ai = rng.integers(len(prepared['aux_h']), size=RECIPE['auxiliary_batch_size'])\n        ids = torch.as_tensor(ii, device=device, dtype=torch.long)\n        aids = torch.as_tensor(ai, device=device, dtype=torch.long)\n        history = prepared['fit_h'][ids].long()\n        candidates = prepared['fit_i'][ids].long()\n        negatives = rng.choice(prepared['n_items'], size=(len(ai), RECIPE['negatives']),\n                               p=prepared['exposure_probability'])\n        negative_ids = torch.as_tensor(negatives, device=device, dtype=torch.long)\n        aux_candidates = torch.cat([prepared['aux_target'][aids, None].long(), negative_ids], dim=1)\n        aux_history = prepared['aux_h'][aids].long()\n        optimizer.zero_grad()\n        logits, penalty = model.forward_and_penalty(history, candidates)\n        aux_logits, aux_penalty = model.forward_and_penalty(aux_history, aux_candidates)\n        aux_labels = torch.zeros_like(aux_logits)\n        aux_labels[:, 0] = 1\n        loss = (nn.functional.binary_cross_entropy_with_logits(logits, prepared['fit_y'][ids])\n                + RECIPE['auxiliary_weight']*nn.functional.binary_cross_entropy_with_logits(aux_logits, aux_labels)\n                + RECIPE['sampled_vector_regularization']*(penalty+RECIPE['auxiliary_weight']*aux_penalty))\n        if not torch.isfinite(loss):\n            raise FloatingPointError('Nonfinite loss')\n        loss.backward()\n        optimizer.step()\n        if (step+1) % 200 == 0:\n            print(json.dumps(dict(seed=seed, step=step+1, loss=float(loss.detach()),\n                                  elapsed_seconds=time.monotonic()-started)), flush=True)\n    def predict(member):\n        result = []\n        with torch.no_grad():\n            for offset in range(0, len(prepared['val_h']), 1024):\n                h = prepared['val_h'][offset:offset+1024].long()\n                i = prepared['val_i'][offset:offset+1024].long()\n                result.append(member(h, i).sigmoid().cpu().numpy().ravel())\n        return np.concatenate(result)\n    pred = predict(model)\n    state = {key: value.detach().cpu() for key, value in model.state_dict().items()}\n    torch.save(state, output/f'seed{seed}.pt')\n    replay = BagFactor(prepared['n_items'], RECIPE['dimensions']).to(device)\n    replay.load_state_dict(torch.load(output/f'seed{seed}.pt', weights_only=True))\n    assert np.array_equal(pred, predict(replay))\n    np.save(output/f'seed{seed}_pred.npy', pred)\n    print(json.dumps(dict(seed=seed, checkpoint_replay='exact_all_rows',\n                          elapsed_seconds=time.monotonic()-started)), flush=True)\n    del model, replay, optimizer\n    gc.collect()\n    torch.cuda.empty_cache()\n    return pred\n\n\ndef run(root, output):\n    root, output = Path(root), Path(output)\n    assert torch.cuda.is_available(), 'GPU required; refusing a long CPU fallback'\n    output.mkdir(parents=True, exist_ok=False)\n    torch.set_num_threads(RECIPE['cpu_threads'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    device = torch.device('cuda:0')\n    started = time.monotonic()\n    gpu = dict(cuda_available=True, name=torch.cuda.get_device_name(0),\n               torch_version=torch.__version__, memory_bytes=torch.cuda.get_device_properties(0).total_memory,\n               nvidia_smi=subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True))\n    frozen = dict(recipe=RECIPE, gpu=gpu, source_sha256=digest(__file__),\n                  inputs=EXPECTED, fresh_holdout=False, official_auc=None,\n                  note='Fixed five seeds, matched single-seed GPU control; no seed selection or per-seed validation scoring')\n    (output/'frozen.json').write_text(json.dumps(frozen, indent=2))\n    print(json.dumps(frozen), flush=True)\n    data, stamps = load(root)\n    reports = []\n    for day in [6, 5]:\n        day_out = output/f'day{day}'\n        day_out.mkdir()\n        with np.load(root/f'val{day}.npz') as z:\n            validation = {key: z[key].copy() for key in z.files}\n        cutoff = int(data['timestamp'][0]) + day*86400\n        prepared = prepare(data, stamps, validation, cutoff)\n        hashes = verify_prepared(prepared, day)\n        (day_out/'prepared_array_hashes.json').write_text(json.dumps(hashes, indent=2))\n        print(json.dumps(dict(stage='prepared', day=day, fitting_events=len(prepared['fit_h']),\n                              auxiliary_clicks=len(prepared['aux_h']))), flush=True)\n        for name in ['val_h', 'val_i']:\n            np.save(day_out/f'{name}.npy', prepared[name])\n        for name in ['fit_h', 'fit_i', 'fit_y', 'aux_h', 'aux_target', 'val_h', 'val_i']:\n            prepared[name] = torch.as_tensor(prepared[name], device=device)\n        predictions = [one_seed(prepared, seed, day_out, device) for seed in RECIPE['seeds']]\n        control_raw = predictions[0]\n        bag_raw = np.mean(np.stack(predictions).astype('float64'), axis=0)\n        rank = lambda x: rankdata(x, method='average')/(len(x)+1)\n        blend = lambda p: (1-RECIPE['blend_weight'])*rank(validation['baseline']) + RECIPE['blend_weight']*rank(p)\n        control, bag = blend(control_raw), blend(bag_raw)\n        def metrics(values):\n            return dict(ordinary_auc=float(roc_auc_score(validation['y'], values)),\n                        adjusted_auc=float(roc_auc_score(validation['y'], values, sample_weight=validation['weights'])))\n        record = dict(day=day, baseline=metrics(validation['baseline']), control=metrics(control),\n                      bag=metrics(bag), raw_control=metrics(control_raw), raw_bag=metrics(bag_raw),\n                      all_five_checkpoint_replays='exact_all_rows', promoted=False)\n        record['adjusted_gain'] = record['bag']['adjusted_auc']-record['baseline']['adjusted_auc']\n        record['adjusted_gain_vs_control'] = record['bag']['adjusted_auc']-record['control']['adjusted_auc']\n        record['development_gate'] = (record['adjusted_gain'] >= RECIPE['minimum_primary_adjusted_gain']\n                                      and record['adjusted_gain_vs_control'] > 0\n                                      and record['bag']['ordinary_auc'] >= record['baseline']['ordinary_auc'])\n        np.savez_compressed(day_out/'predictions.npz', y=validation['y'], events=np.repeat(validation['indices'], 10),\n                            weights=validation['weights'], baseline=validation['baseline'],\n                            control_raw=control_raw, bag_raw=bag_raw, control=control, bag=bag)\n        (day_out/'report.json').write_text(json.dumps(record, indent=2))\n        reports.append(record)\n        print(json.dumps(record), flush=True)\n        del prepared, predictions\n        gc.collect()\n        torch.cuda.empty_cache()\n        if not record['development_gate']:\n            break\n    summary = dict(reports=reports, day5_run=len(reports)==2, target_achieved=False,\n                   local_adjusted_target_reached=reports[0]['bag']['adjusted_auc'] >= 0.83,\n                   promoted=False, csv_ready=False, official_auc=None, fresh_holdout=False,\n                   elapsed_seconds=time.monotonic()-started)\n    (output/'summary.json').write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary), flush=True)\n    return summary\n",encoding="utf-8")
assert hashlib.sha256(module_path.read_bytes()).hexdigest()=='02b53689d03e493f76845f29837150cc9d5d2372be6910b5b44145f5468298e1'
spec=importlib.util.spec_from_file_location("tied_history_gpu_bag",module_path)
bag_module=importlib.util.module_from_spec(spec)
spec.loader.exec_module(bag_module)
root=Path("/kaggle/input/datasets/riazahmadbutt/techarena-neural-causal-histories")
output=Path("/kaggle/working/ctr-tied-history-bag-001")
summary=bag_module.run(root,output)


In [ ]:
import zipfile,hashlib,json
from pathlib import Path
models={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in output.rglob("*.pt")}
(output/"model_sha256.json").write_text(json.dumps(models,indent=2))
for name in ["audit","models"]:
    archive=Path("/kaggle/working")/(output.name+"-"+name+".zip")
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for p in output.rglob("*"):
            if p.is_file() and ((name=="models" and p.suffix==".pt") or (name=="audit" and (p.suffix==".json" or p.name.endswith("_pred.npy") or p.name=="predictions.npz"))): z.write(p,p.relative_to(output))
    print(archive.name,archive.stat().st_size,hashlib.sha256(archive.read_bytes()).hexdigest())
print("LOCAL ADJUSTED:",summary["reports"][0]["bag"]["adjusted_auc"])
print("Experiment bundles only; no competition CSV")
